# 07 — Multi-Agent: Graph Loops

**Stage 7 of the workshop.** A write/review/revise feedback loop using `GraphBuilder`, with a deterministic (non-LLM) quality-check node and explicit loop guards.

## Problem

One agent, one system prompt, one toolset gets unwieldy fast. Splitting into specialized agents that hand off work keeps each one simple — but a plain sequential Workflow can't express "go back and try again if the result isn't good enough." That's what Graph adds.

## Concept

```
Workflow:  A → B → C → D          (deterministic, you wrote the order)
Graph:     A → (B|C) → D           (dynamic routing, conditions decide)
```
`GraphBuilder` adds *conditions* on edges — a node's result decides which edge fires next, which is how a revise-loop becomes possible. Nodes don't have to be LLM agents either: `QualityChecker` here is a plain Python class implementing `MultiAgentBase`, no model call at all.

**When would you NOT use a graph?** If the steps are always the same order with no branching, Workflow is simpler and has nothing to debug — graphs earn their complexity when the *next step depends on evaluating the last one's output*.

## Architecture

```
      writer (LLM agent, writes/revises a blurb)
        │
        ▼
 quality_checker (deterministic, no LLM — checks word count)
        │
   ┌────┴─────┐
   │          │
REVISE     APPROVED
   │          │
   ▼          ▼
 writer   finalizer (LLM agent, extracts final blurb)
(loop back)
```

Loop guards: `max_node_executions(6)`, `execution_timeout(60)`, `reset_on_revisit(True)` — without these, a condition that never resolves to APPROVED loops forever.

## Step 1 — Model and imports

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

from model_provider import get_model
from strands import Agent
from strands.agent.agent_result import AgentResult
from strands.multiagent import GraphBuilder
from strands.multiagent.base import MultiAgentBase, MultiAgentResult, NodeResult
from strands.types.content import ContentBlock

model = get_model()

# Extra headroom for writer/finalizer generations on local Ollama —
# think mode is already disabled by default in model_provider.get_model().
model.update_config(max_tokens=1024)


## Step 2 — Writer and Finalizer agents

In [2]:
writer = Agent(
    model=model,
    name="writer",
    system_prompt=(
        "You write short marketing blurbs (<=60 words). If given feedback, "
        "revise the previous draft to address it. Output only the blurb."
    ),
)

finalizer = Agent(
    model=model,
    name="finalizer",
    system_prompt=(
        "You receive upstream graph output that may include the original task, "
        "verdicts like APPROVED, and a marketing blurb. Find ONLY the final "
        "marketing blurb (the short promotional text, not instructions or "
        "verdicts) and output it prefixed with 'FINAL: ', nothing else."
    ),
)


## Step 3 — QualityChecker: a deterministic, non-LLM node

Implements `MultiAgentBase` directly instead of wrapping an `Agent` — pure Python logic (word count) decides APPROVED vs REVISE.

In [4]:
class QualityChecker(MultiAgentBase):
    """Deterministic node: no LLM. Approves once text is long enough."""

    def __init__(self, min_words: int = 15):
        super().__init__()
        self.min_words = min_words

    def __call__(self, task, **kwargs) -> MultiAgentResult:
        text = task if isinstance(task, str) else str(task)
        approved = len(text.split()) >= self.min_words
        verdict = "APPROVED" if approved else "REVISE: too short, add more detail"
        result = AgentResult(
            stop_reason="end_turn",
            message={"role": "assistant", "content": [ContentBlock(text=verdict)]},
            metrics=None,
            state={},
        )
        return MultiAgentResult(
            results={"quality_checker": NodeResult(result=result)},
            status="completed" if approved else "needs_revision",
        )

    async def invoke_async(self, task, invocation_state=None, **kwargs) -> MultiAgentResult:
        return self.__call__(task, **kwargs)


## Step 4 — Build the graph

Edges carry conditions (lambdas inspecting the previous node's result); loop guards cap runaway execution.

In [5]:
def build_graph():
    checker = QualityChecker(min_words=15)
    builder = GraphBuilder()
    builder.add_node(writer, "writer")
    builder.add_node(checker, "quality_checker")
    builder.add_node(finalizer, "finalizer")

    builder.add_edge("writer", "quality_checker")
    builder.add_edge(
        "quality_checker",
        "writer",
        condition=lambda state: "REVISE" in str(state.results.get("quality_checker")),
    )
    builder.add_edge(
        "quality_checker",
        "finalizer",
        condition=lambda state: "APPROVED" in str(state.results.get("quality_checker")),
    )

    builder.set_max_node_executions(6)
    builder.set_execution_timeout(60)
    builder.reset_on_revisit(True)
    builder.set_entry_point("writer")
    return builder.build()


## Step 5 — Run it

In [7]:
graph = build_graph()
result = graph("Write a blurb for a local Ollama-powered coding agent.")
print(result)

Unlock frictionless coding locally with our Ollama-powered agent. Run entirely on your machine for zero latency, maximum privacy, and instant context awareness. Empower your workflow—write better code faster, right now.FINAL: Boost your productivity with our Ollama-powered coding assistant, locally running and ready to help you write better code in seconds.writer: Unlock frictionless coding locally with our Ollama-powered agent. Run entirely on your machine for zero latency, maximum privacy, and instant context awareness. Empower your workflow—write better code faster, right now.
quality_checker: quality_checker: APPROVED
finalizer: FINAL: Boost your productivity with our Ollama-powered coding assistant, locally running and ready to help you write better code in seconds.


## Failure mode to know about

A graph with no execution guard can loop forever if the condition never resolves to "done" — `max_node_executions` and `execution_timeout` aren't optional safety theater, they're the only thing standing between a bug and an infinite bill (doubly true once this runs against a paid model instead of free local Ollama).